# Gold — fact_morbilidad

Tabla de hechos de morbilidad. Fuente: 3 tablas MSPAS Silver.

**Grain:** un registro por `(anio, departamento, municipio, cie_10_norm, grupo_etario, sexo, source_file)`.

| Columna | Tipo | FK |
|---|---|---|
| `tiempo_sk` | INT | `dim_tiempo` |
| `geografia_sk` | INT | `dim_geografia` (nivel_geo='municipio') |
| `causa_sk` | INT | `dim_causa` |
| `demografia_sk` | INT | `dim_demografia` |
| `fuente_sk` | INT | `dim_fuente` |
| `casos_total` | BIGINT | — |

**Nota:** Si geography join falla (municipio sin match en Silver), `geografia_sk` queda NULL — no se descartan filas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SILVER_SCHEMA = 'stage_silver_ss2'
GOLD_SCHEMA   = 'gold_ss2'
WRITE_MODE    = 'overwrite'

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return 'manual'

RUN_ID = get_job_run_id()
logger.info(f'Setup OK — run_id={RUN_ID}')

## Carga de Silver MSPAS y dimensiones

In [0]:
_MSPAS = [
    'dbx_primeras_causas_de_morbilidad_2015_a_2025',
    'dbx_morbilidad_enfermedades_cronicas_2015_a_2025',
    'dbx_morbilidad_grupo_materno_infantil_2012_a_2025',
]

_SILVER_COLS = [
    'anio', 'departamento_oficial', 'municipio',
    'cie_10_norm', 'grupo_etario', 'sexo',
    'source_system', 'source_file', 'casos_total'
]

df_mspas = reduce(lambda a, b: a.union(b), [
    spark.table(f'{SILVER_SCHEMA}.{t}').select(*_SILVER_COLS)
    for t in _MSPAS
])
logger.info(f'MSPAS Silver (union 3 tablas): {df_mspas.count():,} filas')

dim_tiempo     = spark.table(f'{GOLD_SCHEMA}.dim_tiempo')
dim_geografia  = spark.table(f'{GOLD_SCHEMA}.dim_geografia')
dim_causa      = spark.table(f'{GOLD_SCHEMA}.dim_causa')
dim_demografia = spark.table(f'{GOLD_SCHEMA}.dim_demografia')
dim_fuente     = spark.table(f'{GOLD_SCHEMA}.dim_fuente')
logger.info('Dimensiones cargadas')

## Join con dim_tiempo

In [0]:
df = df_mspas.join(
    dim_tiempo.select('tiempo_sk', 'anio'),
    on='anio',
    how='left'
)
logger.info(f'Despues de join tiempo: {df.count():,} filas')

## Join con dim_geografia (nivel municipio)

In [0]:
dim_geo_muni = (
    dim_geografia
    .filter(F.col('nivel_geo') == 'municipio')
    .select(
        F.col('geografia_sk'),
        F.col('departamento').alias('_geo_depto'),
        F.col('municipio').alias('_geo_muni')
    )
)

df = df.join(
    dim_geo_muni,
    (F.col('departamento_oficial') == F.col('_geo_depto')) &
    (F.col('municipio') == F.col('_geo_muni')),
    how='left'
).drop('_geo_depto', '_geo_muni')

sin_geo = df.filter(F.col('geografia_sk').isNull()).count()
logger.info(f'Join geografia OK. Filas sin match: {sin_geo:,}')

## Join con dim_causa

In [0]:
dim_causa_sel = dim_causa.select(
    F.col('causa_sk'),
    F.col('codigo_origen').alias('_causa_codigo')
)

df = df.join(
    dim_causa_sel,
    F.col('cie_10_norm') == F.col('_causa_codigo'),
    how='left'
).drop('_causa_codigo')

sin_causa = df.filter(F.col('causa_sk').isNull()).count()
logger.info(f'Join causa OK. Filas sin match: {sin_causa:,}')

## Join con dim_demografia

In [0]:
dim_demo_sel = dim_demografia.select(
    F.col('demografia_sk'),
    F.col('sexo').alias('_demo_sexo'),
    F.col('grupo_etario_label').alias('_demo_grupo')
)

# sexo=NULL no matchea en Spark (null==null es false). Filas sin sexo se mapean
# al placeholder (Ambos, Sin desagregar) para evitar FK nulos.
df = (
    df
    .withColumn('_sexo_join',  F.coalesce(F.col('sexo'), F.lit('Ambos')))
    .withColumn('_grupo_join', F.when(F.col('sexo').isNull(), F.lit('Sin desagregar'))
                                .otherwise(F.coalesce(F.col('grupo_etario'), F.lit('Sin desagregar'))))
)

df = df.join(
    dim_demo_sel,
    (F.col('_sexo_join') == F.col('_demo_sexo')) &
    (F.col('_grupo_join') == F.col('_demo_grupo')),
    how='left'
).drop('_demo_sexo', '_demo_grupo', '_sexo_join', '_grupo_join')

sin_demo = df.filter(F.col('demografia_sk').isNull()).count()
logger.info(f'Join demografia OK. Filas sin match: {sin_demo:,}')

## Join con dim_fuente

In [0]:
dim_fuente_sel = dim_fuente.select(
    F.col('fuente_sk'),
    F.col('source_system').alias('_fs_sys'),
    F.col('source_file').alias('_fs_file')
)

df = df.join(
    dim_fuente_sel,
    (F.col('source_system') == F.col('_fs_sys')) &
    (F.col('source_file') == F.col('_fs_file')),
    how='left'
).drop('_fs_sys', '_fs_file')

sin_fuente = df.filter(F.col('fuente_sk').isNull()).count()
logger.info(f'Join fuente OK. Filas sin match: {sin_fuente:,}')

## Seleccion final y escritura

In [0]:
df_fact = df.select(
    'tiempo_sk',
    'geografia_sk',
    'causa_sk',
    'demografia_sk',
    'fuente_sk',
    'casos_total'
)

total = df_fact.count()
logger.info(f'fact_morbilidad: {total:,} filas')

df_fact.write \
    .format('delta') \
    .mode(WRITE_MODE) \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{GOLD_SCHEMA}.fact_morbilidad')

logger.info(f'Escrito -> {GOLD_SCHEMA}.fact_morbilidad')

## Validacion

In [0]:
df_val = spark.table(f'{GOLD_SCHEMA}.fact_morbilidad')
print(f'Total filas: {df_val.count():,}')

print('\n-- NULLs en claves foraneas --')
for col in ['tiempo_sk', 'geografia_sk', 'causa_sk', 'demografia_sk', 'fuente_sk']:
    n = df_val.filter(F.col(col).isNull()).count()
    print(f'  {col}: {n:,} NULLs')

print('\n-- Query de prueba: top 10 causas por casos --')
(
    df_val
    .join(spark.table(f'{GOLD_SCHEMA}.dim_causa').select('causa_sk', 'descripcion'), 'causa_sk')
    .join(spark.table(f'{GOLD_SCHEMA}.dim_tiempo').select('tiempo_sk', 'anio'), 'tiempo_sk')
    .groupBy('anio', 'descripcion')
    .agg(F.sum('casos_total').alias('total_casos'))
    .orderBy(F.desc('total_casos'))
    .limit(10)
    .show(truncate=False)
)